# GLiNER / GLiREL Benchmark — Plan C

Standalone benchmark notebook. Does **not** modify any existing pipeline file — read-only against the live graph (`KnowledgeGarden()`), and only calls the current extraction pipeline (`extract_entities_and_relations`) as-is.

**Goal:** determine whether GLiNER (NER) + GLiREL (relation extraction) — off-the-shelf, zero training — can replace the current Gemini/Groq LLM extraction calls at zero cost / zero rate limit, without meaningful quality loss. See `IMPROVEMENTS.md`'s "Extraction & Entity-Resolution Accuracy" section for the research pass that motivated this (bottom line: benchmark GLiNER/GLiREL before considering any fine-tune).

**Decision rule (from the research pass):**
- GLiNER+GLiREL F1 within ~10% of the current LLM pipeline → ready to adopt.
- Gap >15-20% → not ready; revisit fine-tuning only once a stable 500-1,500+ example labeled dataset of our own confirmed graph data exists.


## Step 1 — Setup

This notebook needs, in addition to this repo's normal dependencies:
- A running FalkorDB with real ingested episodes (`docker` — see `README.md`'s setup section)
- `GROQ_API_KEY` (or whatever `LLM_PROVIDER`/`EXTRACTION_LLM_PROVIDER` you use) set in `.env`, since Step 4 re-runs the current LLM pipeline for comparison
- `gliner` and `glirel` installed (Step 2 below) — **not yet installed in this environment**, install before running past Step 2


In [ ]:
import sys
sys.path.append('..')


In [ ]:
import json
from collections import defaultdict

from core.graph import KnowledgeGarden
from llm_clients import LLMClient
from enrichment.extractor import extract_entities_and_relations
from enrichment.resolver import normalize

# Keep this in sync with prompts.py's EXTRACTION_PROMPT / RELATION_EXTRACTION_PROMPT
# allowed vocabulary — used as the zero-shot label set for GLiNER/GLiREL below,
# so all three pipelines are being asked to extract into the exact same schema.
NODE_TYPES = ["person", "service", "team", "tool", "concept", "event", "document"]
RELATION_TYPES = [
    "MEMBER_OF", "OWNS", "DEPENDS_ON", "USES", "REPORTED", "RESOLVED_BY",
    "MENTIONED_IN", "ATTENDED", "DISCUSSED", "DECIDED", "SENT_TO", "SCHEDULED",
    "AUTHORED", "REFERENCES",
]
# GLiREL's zero-shot label prompts read better as natural-language phrases than
# our SCREAMING_SNAKE_CASE constants (e.g. "member of" vs "MEMBER_OF") — this map
# lets GLiREL predict against readable labels while everything downstream still
# scores against our canonical relation vocabulary.
RELATION_LABEL_TO_CANONICAL = {
    "member of": "MEMBER_OF",
    "owns": "OWNS",
    "depends on": "DEPENDS_ON",
    "uses": "USES",
    "reported": "REPORTED",
    "resolved by": "RESOLVED_BY",
    "mentioned in": "MENTIONED_IN",
    "attended": "ATTENDED",
    "discussed": "DISCUSSED",
    "decided": "DECIDED",
    "sent to": "SENT_TO",
    "scheduled": "SCHEDULED",
    "authored": "AUTHORED",
    "references": "REFERENCES",
}
RELATION_LABELS_FOR_GLIREL = list(RELATION_LABEL_TO_CANONICAL.keys())


## Step 2 — Install & smoke test

Run the install once (not yet installed in this environment as of when this notebook was authored):
```
pip install gliner glirel
```

**Note on GLiREL's API below:** this notebook was authored without the package installed (no network access from the authoring environment to confirm against the live PyPI package / GitHub README) — the `predict_relations(...)` call shape is written to match GLiREL's documented usage as of its ACL 2025 release, but **verify it against `help(GLiREL)` / the installed package's own README first** and adjust if the signature has since changed. Everything downstream (scoring, ground truth) is independent of the exact call shape, so a signature fix here is isolated to this one cell.


In [ ]:
from gliner import GLiNER

gliner_model = GLiNER.from_pretrained("gliner-community/gliner_medium-v2.5")


In [ ]:
# GLiREL is typically distributed as a spaCy pipeline component or a standalone
# model class depending on version — try the standalone class first; if this
# import/API fails, check the installed package's README (`pip show glirel`,
# then look at its usage examples) and adjust this cell only.
from glirel import GLiREL

glirel_model = GLiREL.from_pretrained("jackboyla/glirel_beta")


In [ ]:
# Smoke test on one hand-written sentence before touching real graph data.
smoke_text = "Alice joined the infra team last Monday. The infra team owns the auth service, which depends on Postgres."

smoke_entities = gliner_model.predict_entities(smoke_text, NODE_TYPES, threshold=0.5)
print("GLiNER entities:")
for e in smoke_entities:
    print(f"  {e['text']!r:30} type={e['label']:10} score={e['score']:.2f}")


In [ ]:
# GLiREL needs entity spans as (start_token, end_token, label) triples over a
# tokenized input, not raw character offsets — check the installed version's
# expected tokenization (whitespace split vs spaCy Doc) and adjust `tokens`/
# `ner_spans` construction here if this doesn't match.
tokens = smoke_text.split()

# Build (start, end, label) spans by locating each GLiNER entity's tokens —
# naive whitespace-based alignment, good enough for a smoke test.
def token_span(tokens, entity_text):
    words = entity_text.split()
    for i in range(len(tokens) - len(words) + 1):
        if tokens[i:i+len(words)] == words:
            return i, i + len(words)
    return None

ner_spans = []
for e in smoke_entities:
    span = token_span(tokens, e["text"])
    if span:
        ner_spans.append([span[0], span[1], e["label"]])
print("NER spans for GLiREL:", ner_spans)

smoke_relations = glirel_model.predict_relations(
    tokens, RELATION_LABELS_FOR_GLIREL, threshold=0.0, ner=ner_spans, top_k=1
)
print("\nGLiREL relations:")
for r in smoke_relations:
    print(f"  {r}")


## Step 3 — Pull real episodes + build ground-truth scaffold

Pulls up to 30 real already-ingested episodes directly via Cypher (read-only — `KnowledgeGarden` has no `get_all_episodes()` helper today, and this notebook intentionally doesn't add one to `core/graph.py` per Plan C's "don't touch existing files" constraint).


In [ ]:
kg = KnowledgeGarden()

result = kg._graph.query("MATCH (ep:Episode) RETURN ep LIMIT 30")
episodes = [kg._episode_from_props(r[0].properties) for r in result.result_set]
print(f"Pulled {len(episodes)} episodes")


In [ ]:
# Run the CURRENT pipeline once now (not per-scoring-run below) so its output
# is available right next to each episode's text for you to eyeball while
# filling in ground truth — saves re-reading the source text separately.
llm_client = LLMClient()  # uses LLM_PROVIDER / EXTRACTION_LLM_PROVIDER from .env

current_llm_results = {}
for ep in episodes:
    current_llm_results[ep.id] = extract_entities_and_relations(ep.text, llm_client)


In [ ]:
for ep in episodes:
    resp = current_llm_results[ep.id]
    print(f"── episode {ep.id} " + "─" * 40)
    print(ep.text)
    print("\ncurrent LLM pipeline extracted:")
    print("  nodes:", [(n.name, n.type) for n in resp.nodes])
    print("  edges:", [(e.source, e.relation, e.target) for e in resp.edges])
    print()


### Fill in ground truth here

For each episode above, add the entities/relations that **should** have been extracted (your own judgment — not what either pipeline produced). Use the exact allowed vocabulary (`NODE_TYPES`/`RELATION_TYPES` above). Leave an episode's lists empty if it genuinely has nothing to extract — don't skip the key entirely, or it'll be excluded from scoring silently.

Format per episode: `{"entities": [(name, type), ...], "relations": [(source, relation, target), ...]}`


In [ ]:
GROUND_TRUTH = {
    # ep.id: {"entities": [("Alice", "person"), ("infra team", "team")], "relations": [("Alice", "MEMBER_OF", "infra team")]},
}

# Scaffold: pre-fills every pulled episode with an empty template so none get
# silently skipped — fill in the actual lists above or edit these in place.
for ep in episodes:
    GROUND_TRUTH.setdefault(ep.id, {"entities": [], "relations": []})

print(f"{len(GROUND_TRUTH)} episodes ready for ground truth — fill in the dict above, then re-run this cell's print to sanity check counts.")
print(f"Currently filled in (non-empty): {sum(1 for v in GROUND_TRUTH.values() if v['entities'] or v['relations'])}/{len(GROUND_TRUTH)}")


## Step 4 — Run three pipelines, score against ground truth


In [ ]:
def run_gliner_glirel(text: str):
    entities = gliner_model.predict_entities(text, NODE_TYPES, threshold=0.5)
    tokens = text.split()
    ner_spans = []
    for e in entities:
        span = token_span(tokens, e["text"])
        if span:
            ner_spans.append([span[0], span[1], e["label"]])

    relations = []
    if ner_spans:
        raw_relations = glirel_model.predict_relations(
            tokens, RELATION_LABELS_FOR_GLIREL, threshold=0.0, ner=ner_spans, top_k=1
        )
        for r in raw_relations:
            # NOTE: adjust these key names once you've confirmed GLiREL's actual
            # output shape in Step 2 — written defensively (.get with a couple
            # of plausible key names) since the exact shape wasn't verified.
            head_text = r.get("head_text") or r.get("head")
            tail_text = r.get("tail_text") or r.get("tail")
            label = r.get("label")
            canonical = RELATION_LABEL_TO_CANONICAL.get(label)
            if head_text and tail_text and canonical:
                relations.append((head_text, canonical, tail_text))

    return (
        [(e["text"], e["label"]) for e in entities],
        relations,
    )


In [ ]:
def run_current_llm(ep_id: str):
    resp = current_llm_results[ep_id]
    return (
        [(n.name, n.type) for n in resp.nodes],
        [(e.source, e.relation, e.target) for e in resp.edges],
    )


In [ ]:
def _norm_entity(pair):
    name, type_ = pair
    return (normalize(name), type_.lower())

def _norm_relation(triple):
    source, relation, target = triple
    return (normalize(source), relation.upper(), normalize(target))

def score(predicted: list, ground_truth: list, norm_fn) -> dict:
    """Micro precision/recall/F1 — set-based exact match after normalization."""
    pred_set = {norm_fn(p) for p in predicted}
    gt_set = {norm_fn(g) for g in ground_truth}
    tp = len(pred_set & gt_set)
    fp = len(pred_set - gt_set)
    fn = len(gt_set - pred_set)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1}

def aggregate(per_episode_counts: list) -> dict:
    """Micro-average across all episodes (sum tp/fp/fn first, then compute
    P/R/F1 once) rather than averaging per-episode F1 — standard for small,
    uneven-length benchmark sets like this one."""
    tp = sum(c["tp"] for c in per_episode_counts)
    fp = sum(c["fp"] for c in per_episode_counts)
    fn = sum(c["fn"] for c in per_episode_counts)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1}


In [ ]:
labeled_episodes = [ep for ep in episodes if GROUND_TRUTH[ep.id]["entities"] or GROUND_TRUTH[ep.id]["relations"]]
skipped = len(episodes) - len(labeled_episodes)
if skipped:
    print(f"NOTE: {skipped}/{len(episodes)} episodes have no ground truth filled in yet — excluded from scoring below, not silently counted as correct.")

llm_entity_scores, llm_relation_scores = [], []
gliner_entity_scores, gliner_relation_scores = [], []

for ep in labeled_episodes:
    gt = GROUND_TRUTH[ep.id]

    llm_entities, llm_relations = run_current_llm(ep.id)
    llm_entity_scores.append(score(llm_entities, gt["entities"], _norm_entity))
    llm_relation_scores.append(score(llm_relations, gt["relations"], _norm_relation))

    g_entities, g_relations = run_gliner_glirel(ep.text)
    gliner_entity_scores.append(score(g_entities, gt["entities"], _norm_entity))
    gliner_relation_scores.append(score(g_relations, gt["relations"], _norm_relation))

results = {
    "current_llm": {
        "entities": aggregate(llm_entity_scores),
        "relations": aggregate(llm_relation_scores),
    },
    "gliner_glirel": {
        "entities": aggregate(gliner_entity_scores),
        "relations": aggregate(gliner_relation_scores),
    },
}
print(json.dumps(results, indent=2))


## Step 5 — Decision summary


In [ ]:
def f1_gap_pct(a: float, b: float) -> float:
    """Relative gap between two F1 scores, as a percent of the higher one."""
    hi, lo = max(a, b), min(a, b)
    return 0.0 if hi == 0 else (hi - lo) / hi * 100

entity_gap = f1_gap_pct(results["current_llm"]["entities"]["f1"], results["gliner_glirel"]["entities"]["f1"])
relation_gap = f1_gap_pct(results["current_llm"]["relations"]["f1"], results["gliner_glirel"]["relations"]["f1"])

print(f"Benchmark set size: {len(labeled_episodes)} labeled episodes ({skipped} skipped, no ground truth)\n")
print(f"Entity F1   — current LLM: {results['current_llm']['entities']['f1']:.3f} | GLiNER: {results['gliner_glirel']['entities']['f1']:.3f} | gap: {entity_gap:.1f}%")
print(f"Relation F1 — current LLM: {results['current_llm']['relations']['f1']:.3f} | GLiREL: {results['gliner_glirel']['relations']['f1']:.3f} | gap: {relation_gap:.1f}%")
print()

for label, gap in [("Entity", entity_gap), ("Relation", relation_gap)]:
    if gap <= 10:
        verdict = "READY TO ADOPT — within ~10% F1 of the LLM pipeline"
    elif gap <= 20:
        verdict = "BORDERLINE — 10-20% gap, re-run with a larger ground-truth set before deciding"
    else:
        verdict = "NOT READY — >20% gap, defer to a future fine-tuning pass (see IMPROVEMENTS.md)"
    print(f"{label}: {verdict}")
